# Bloque 3: Ensamblando el Agente con LangGraph

Hasta ahora, tenemos un "Cerebro" (el LLM) y unas "Manos" (las Herramientas). LangGraph es el "Sistema Nervioso" que los conecta.

LangGraph nos permite construir agentes definiendo un **Grafo de Estado** (StateGraph) donde:
1. **El Estado (State):** Es la memoria compartida (en nuestro caso, el historial de mensajes).
2. **Los Nodos (Nodes):** Son funciones de Python que hacen el trabajo (ej. pensar, ejecutar una herramienta).
3. **Las Aristas (Edges):** Son las reglas que deciden qué nodo se ejecuta después (flujo de control).

Implementaremos la arquitectura **ReAct (Reason + Act)**, creando un ciclo donde el modelo razona, decide usar una herramienta, observa el resultado, y vuelve a razonar.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Agregamos la ruta padre para poder importar desde la carpeta 'app'
sys.path.append(os.path.abspath('..'))
load_dotenv(dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from langgraph.graph import END
# Importamos las herramientas que ya programamos (incluidas las de agenda)
from app.tools_notebooks import (
    buscar_cliente,
    obtener_precio_accion,
    consultar_inventario,
    consultar_agenda_dia,
    agendar_reunion,
)

# Herramientas del nodo principal (cliente, precio, inventario) y de agenda
tools = [buscar_cliente, obtener_precio_accion, consultar_inventario]
tools_agenda = []

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
# El chatbot tiene acceso a todas las herramientas para decidir cuál usar
llm_with_tools = llm.bind_tools(tools + tools_agenda)

print("✅ Modelo y herramientas configuradas.")

## 1. Definiendo el Estado y los Nodos

Utilizaremos `MessagesState`, un estado predefinido por LangGraph que básicamente es un diccionario con una clave `"messages"` que contiene la lista de la conversación.

* **Nodo Agente:** Llama al LLM para que evalúe el estado y decida qué hacer.
* **Nodo sql_tools:** Ejecuta las herramientas de cliente, precio e inventario.
* **Nodo de agenda:** Deberás crearlo en el ejercicio (ver última celda).

In [ ]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage

# Nodo 1: El cerebro del agente
def chatbot_node(state: MessagesState):
    """
    Este nodo es el 'Cerebro'. Toma el historial de mensajes actual, 
    le añade instrucciones de sistema y le pide al LLM que decida el siguiente paso.
    """
    mensajes = [
        SystemMessage(content="""Eres un asistente corporativo altamente eficiente. 
        Tienes acceso a bases de datos de inventario, clientes, y herramientas de agenda.
        Usa las herramientas a tu disposición para responder a las solicitudes del usuario.
        Si no sabes la respuesta o la herramienta falla, admítelo y pide aclaraciones.
        La fecha actual es 26 de febrero de 2026.""")
    ] + state["messages"]
    
    # Invocamos al modelo
    response = llm_with_tools.invoke(mensajes)
    
    # Devolvemos el nuevo mensaje generado (LangGraph lo agregará a la lista automáticamente)
    return {"messages": [response]}

# Nodo que ejecuta las herramientas de cliente, precio e inventario
sql_node = ToolNode(tools=tools)

# Crea aquí el nodo de agenda (usa tools_agenda de la primera celda):
# agenda_node = ToolNode(tools=tools_agenda)

print("✅ Nodos definidos.")

In [ ]:
# Nombres de las herramientas SQL (para enrutar al nodo "sql_tools")
# Usa la misma idea para definir los nombres de las herramientas de agenda.
SQL_TOOL_NAMES = {"buscar_cliente", "obtener_precio_accion", "consultar_inventario"}
# AGENDA_TOOL_NAMES = {...}  

def route_after_agent(state: MessagesState):
    """
    Decide a qué nodo ir después del agente según las tool_calls del último mensaje.
    """
    ultimo_mensaje = state["messages"][-1]
    tool_calls = getattr(ultimo_mensaje, "tool_calls", None)

    if not tool_calls:
        return END

    nombres_herramientas = {llamada["name"] for llamada in ultimo_mensaje.tool_calls}

    # Template: si alguna herramienta es de SQL -> ir a "sql_tools"
    alguna_es_de_sql = any(nombre in SQL_TOOL_NAMES for nombre in nombres_herramientas)
    if alguna_es_de_sql:
        return "sql_tools"

    # Añade aquí la lógica para agenda (compara con AGENDA_TOOL_NAMES y return "agenda_tools")
    # alguna_es_de_agenda = ...
    # if alguna_es_de_agenda:
    #     return "agenda_tools"

    return END

## 2. Construyendo el Grafo y las Aristas (Edges)

Aquí definimos el flujo: inicio → agente → (según `route_after_agent`) ir a `sql_tools`, a `agenda_tools` o a END. Después de ejecutar herramientas, siempre se vuelve al agente.

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(MessagesState)

workflow.add_node("agent", chatbot_node)
workflow.add_node("sql_tools", sql_node)
# Añade el nodo de agenda: workflow.add_node("agenda_tools", agenda_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_after_agent,
                               {
        "sql_tools": "sql_tools",
        END: END
    })
workflow.add_edge("sql_tools", "agent")
# Añade la arista de vuelta desde agenda_tools al agente

graph = workflow.compile()
print("✅ Grafo compilado exitosamente.")

## 3. Poniendo al Agente a Trabajar

Vamos a hacerle una pregunta al agente que lo obligue a usar la herramienta de base de datos que programamos, específicamente para ver cómo maneja el problema de **Ambigüedad** con el nombre "Carlos".

Usaremos el método `.stream()` para ver cómo la información fluye de nodo a nodo en tiempo real.

In [ ]:
from langchain_core.messages import HumanMessage

# La petición inicial del usuario
mensaje = HumanMessage(content="¿Me puedes dar el correo de Carlos Ruiz?")

print("Iniciando ejecución del grafo...\n")
print("-" * 40)

# stream() nos devuelve actualizaciones de estado cada vez que un nodo termina
for event in graph.stream({"messages": [mensaje]}):
    for node_name, node_state in event.items():
        print(f"--- [Nodo ejecutado: {node_name}] ---")
        
        # Obtenemos el último mensaje generado por este nodo
        ultimo_mensaje = node_state["messages"][-1]
        
        # Si el nodo es el agente y generó una llamada a herramienta:
        if node_name == "agent" and hasattr(ultimo_mensaje, "tool_calls") and ultimo_mensaje.tool_calls:
            for tool_call in ultimo_mensaje.tool_calls:
                print(f"🔧 Herramienta solicitada: {tool_call['name']}")
                print(f"📥 Argumentos: {tool_call['args']}")
                
        # Si el nodo es el agente y dio una respuesta final (texto):
        elif node_name == "agent" and ultimo_mensaje.content:
            print(f"\n🤖 Agente: {ultimo_mensaje.content}\n")
            
        # Si el nodo son las herramientas, mostramos el resultado:
        elif node_name in ("sql_tools", "agenda_tools"):
            print(f"📤 Resultado de la herramienta: {ultimo_mensaje.content[:150]}...\n")
        
        print("-" * 40)

---
## Ejercicio: Integrar el nodo de agenda en el grafo

Sigue estos pasos para que el agente pueda usar también las herramientas de agenda (`consultar_agenda_dia`, `agendar_reunion`).

### Paso 1 – Nodo de agenda (celda 3)
- Crea el nodo que ejecutará las herramientas de agenda, siguiendo el mismo patrón que `sql_node`.
- Usa la variable **`tools_agenda`** que ya está definida en la primera celda.
- Asigna el nodo a una variable llamada **`agenda_node`** (así sabrás dónde usarla en el grafo).

### Paso 2 – Nombres de herramientas de agenda (celda 4)
- Define un conjunto **`AGENDA_TOOL_NAMES`** con los nombres `"consultar_agenda_dia"` y `"agendar_reunion"`, igual que se hizo con **`SQL_TOOL_NAMES`** para las herramientas SQL.

### Paso 3 – Lógica de enrutamiento (celda 4)
- Dentro de `route_after_agent`, después del bloque que comprueba `alguna_es_de_sql`:
  - Crea una variable **`alguna_es_de_agenda`** que compruebe si algún nombre en `nombres_herramientas` está en `AGENDA_TOOL_NAMES` (usa el mismo patrón que `alguna_es_de_sql`).
  - Si `alguna_es_de_agenda` es verdadero, haz **`return "agenda_tools"`**.

### Paso 4 – Registrar el nodo en el grafo (celda 6)
- Añade el nodo de agenda al workflow: **`workflow.add_node("agenda_tools", agenda_node)`** (junto al resto de `add_node`).
- Añade la arista de vuelta al agente: **`workflow.add_edge("agenda_tools", "agent")`** (igual que con `sql_tools`).

### Comprobar
- Vuelve a ejecutar el notebook desde el inicio.
- Prueba una pregunta de agenda, por ejemplo: *"¿Qué tengo en la agenda el 2026-02-26?"* o *"Agenda una reunión con el equipo para el 2026-03-01 a las 10:00"*.